In [ ]:
!pip install langchain
!pip install openai
!pip install faiss-cpu
!pip install pypdf
!pip install streamlit
!pip install tiktoken

In [ ]:
!pip install -U langchain langchain-community langchain-openai
!pip install pypdf

In [ ]:
!pip install sentence-transformers
!pip install -U sentence-transformers transformers torch

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-**************************************"

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

documents = []

folder = "C:/Users/HP/Desktop/rag_document_qa/data/document"

for file in os.listdir(folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        documents.extend(loader.load())

print("Documents Loaded:", len(documents))



from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print("Chunks:", len(docs))



from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()



db = FAISS.from_documents(docs, embeddings)
db.save_local("faiss_index")

print("Vector DB Created")



db = FAISS.load_local("faiss_index", embeddings)
retriever = db.as_retriever()

from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

retriever = db.as_retriever()

llm = ChatOpenAI(temperature=0)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)


query = "What is this document about?"

result = qa.run(query)

print(result)